# Build Constructor Standings
#### Sources
 1. fact_session_results
 1. dim_constructors

#### Output Columns
 1. season
 1. constructor id
 1. constructor name
 1. nationality
 1. race starts
 1. total points
 1. number of wins
 1. number of podiums
 1. standing position


#### Entity Relationship Diagram - Formula1 Gold Schema
![Formula1 Gold Data.png](../../z-course-images/formula1-gold-data-erd.png "Formula1 Gold Data.png")


In [0]:
%sql
CREATE OR REPLACE VIEW formula1.gold.v_constructors_standing
AS
with constructors_session_summary as
(select r.season,
        c.constructor_id,
        c.constructor_name,
        c.nationality,
       COUNT(*) AS race_starts,
       SUM(r.points) AS total_points,
       COUNT_IF(r.is_win) AS number_of_wins,
       COUNT_IF(r.is_podium) AS number_of_podiums
from
formula1.gold.fact_session_results r
join formula1.gold.dim_constructors c
on r.constructor_id = c.constructor_id
GROUP BY r.season,
       c.constructor_id,
        c.constructor_name,
        c.nationality) 
SELECT season,
       constructor_id,
       constructor_name,
       nationality,
       RANK() OVER (PARTITION BY season ORDER BY total_points DESC, number_of_wins DESC) AS standing,
       race_starts,
       total_points,
       number_of_wins,
       number_of_podiums
  FROM constructors_session_summary;


In [0]:
%sql
SELECT * FROM formula1.gold.v_constructors_standing WHERE season = 2025